# Step 01b. The curated gene sets, matched to SomaScan probes

Reads the gene lists in `genes/` and `feature_metadata.txt` from the study download. Writes
`ifn_panel.csv` to `data/run_artifacts/`.

**Objective.** Turn curated HGNC gene symbols into the column names this study measures, once, so
that every later notebook uses the same panel and none of them repeats the matching.

**Why this is a step of its own.** A gene symbol is not a measurement. SomaScan measures SOMAmer
reagents, several of which can target the same protein, and a curated list drawn from transcriptomic
papers contains genes this assay does not carry at all. Both facts have to be stated in numbers
before any module is named, because they set the ceiling on what the panel can show.

**The gene lists.** `genes/` is the same directory, file for file, as in `endotypes-transcriptomics`.

- `ifn-response-genes-28.txt`: the 28 interferon response genes. The same 28 genes are listed in
  Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185 (Supplementary Table S1) and in
  de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682 (Supplemental Figure 3B).
- The two papers define different 25-gene subsets:
  - Kim 2018 (`ifn-response-genes-25-kim2018.txt`): the 28 without EPSTI1, SIGLEC1 and SPATS2L, the
    three genes added from the literature.
  - de Jesus 2020 (`ifn-response-genes-25-stat1-only-dejesus2020.txt`): the 28 without CXCL10, GBP1
    and SOCS1 (`ifn-response-genes-3-stat1-nfkb-dejesus2020.txt`), the genes that have both a STAT1
    and an NF-kB binding site.
- `nfkb-only-control-11-dejesus2020.txt`: 11 NF-kB target genes with no STAT1, STAT2 or TYK1 binding
  site (de Jesus 2020, Methods Step 2 and Supplemental Figure 5). **They are the control.** They are
  immune genes, so they are not a trivial null, and they should not be in the interferon module.

The header of each gene file gives the sources and the differences between the two papers.

**What this replaces.** `proteins/interferon-response-genes.json`, a 37-gene list with no single
primary citation behind it. 17 of its genes are not in the 28 (BST2, CXCL11, DDX58, DTX3L, EIF2AK2,
GBP2, IFI16, IFIH1, IRF7, IRF9, MX2, PARP9, SAMD9, SAMD9L, STAT1, STAT2, XAF1) and 8 of the 28 are
missing from it (DDX60, EPSTI1, HERC6, IFIT5, LAMP3, RTP4, SOCS1, SPATS2L), of which RTP4 is on the
menu and was therefore being dropped.

## Load the gene lists

In [1]:
source("../src/paths.R")
irg28  <- read_gene_set("ifn-response-genes-28.txt")
kim25  <- read_gene_set("ifn-response-genes-25-kim2018.txt")
dj25   <- read_gene_set("ifn-response-genes-25-stat1-only-dejesus2020.txt")
dj3    <- read_gene_set("ifn-response-genes-3-stat1-nfkb-dejesus2020.txt")
nfkb11 <- read_gene_set("nfkb-only-control-11-dejesus2020.txt")
g <- data.frame(set  = rep(c("IRG-28", "NF-kB-11"), c(length(irg28), length(nfkb11))),
                gene = c(irg28, nfkb11), stringsAsFactors = FALSE)
g$kim2018_25          <- ifelse(g$set == "IRG-28", g$gene %in% kim25, NA)
g$dejesus2020_25      <- ifelse(g$set == "IRG-28", g$gene %in% dj25,  NA)
g$dejesus2020_subscore<- ifelse(g$set == "IRG-28",
                                ifelse(g$gene %in% dj3, "3 STAT1/NF-kB", "25 STAT1-only"), "control")
table(g$set)


  IRG-28 NF-kB-11 
      28       11 

**Explanation**
- `source("../src/paths.R")` loads the locations and the two file readers. It defines `ROOT` by
  walking up from this directory until it finds `endotypes-proteomics.yml`, then `raw()`, `coh()`,
  `art()` and `fig()`, which build paths into the study download, `cohorts/`,
  `data/run_artifacts/` and `figures/`. It decides nothing.
- `read_gene_set(name)` reads `genes/<name>`, drops blank lines and lines starting with `#`, and
  returns the unique gene symbols. It is the same function, character for character, as in
  `endotypes-transcriptomics/src/paths.R`.
- `data.frame(set = ..., gene = ...)` makes one row per gene.
  `rep(c("IRG-28", "NF-kB-11"), c(28, 11))` repeats each label as many times as its list is long.
- `kim2018_25` and `dejesus2020_25` are TRUE for the genes in each paper's 25-gene score, and `NA`
  for the control genes, which are in neither.
- `dejesus2020_subscore` is `"3 STAT1/NF-kB"` for CXCL10, GBP1 and SOCS1, `"25 STAT1-only"` for the
  other 25, and `"control"` for the 11.
- `table(g$set)` counts the genes in each list: 28 and 11.

## Match the genes to the SomaScan menu

In [2]:
fm <- read.delim(raw("feature_metadata.txt"), check.names = FALSE)
fm$gene <- sub("_seq\\.[0-9.]+$", "", fm$GeneSymbol)
p <- merge(g, fm[, c("gene", "SeqId", "Target", "TargetFullName", "UniProt", "EntrezGeneID",
                     "GeneSymbol")],
           by = "gene", all.x = TRUE, sort = FALSE)
names(p)[names(p) == "GeneSymbol"] <- "column"
p$on_menu <- !is.na(p$column)
p <- p[order(p$set, p$gene, p$SeqId), ]
c(reagents_on_menu = nrow(fm), gene_symbols = length(unique(fm$gene)),
  unique_uniprot = length(unique(fm$UniProt[nzchar(fm$UniProt)])),
  unique_entrez  = length(unique(fm$EntrezGeneID[nzchar(fm$EntrezGeneID)])))

reagents_on_menu     gene_symbols   unique_uniprot    unique_entrez 
            7288             6399             6401             6396

**Explanation**
- `read.delim(raw("feature_metadata.txt"))` reads the study's own reagent table, one row per
  SOMAmer: `SeqId`, `Target`, `TargetFullName`, `UniProt`, `EntrezGeneID`, `GeneSymbol`.
  `check.names = FALSE` keeps the column names exactly as the file writes them.
- `sub("_seq\\.[0-9.]+$", "", fm$GeneSymbol)` recovers the plain gene symbol. **This is the one
  thing about this file that must be understood.** `GeneSymbol` is already disambiguated for genes
  with more than one reagent: ISG15 appears as `ISG15_seq.14148.2` and `ISG15_seq.14151.4`, while
  single-reagent genes appear bare, as `MX1`. `GeneSymbol` is therefore the name the cohort matrices
  carry as a column, and `"ISG15"` on its own matches no column at all. The regexp strips the
  `_seq.<id>` suffix, so `gene` holds the symbol and `column` holds the measurement.
- `merge(..., by = "gene", all.x = TRUE)` keeps one row per curated gene per reagent, and one row
  with `NA` for a curated gene the assay does not carry. `all.x = TRUE` is what keeps the absent
  genes visible instead of silently dropping them.
- `on_menu` is TRUE where a reagent was found.
- The last line reports the menu itself: SOMAmers, distinct gene symbols, distinct UniProt
  accessions, distinct Entrez IDs. **The count of distinct genes is not the count of reagents**, and
  the difference is the whole reason `n_proteins()` exists in `paths.R`.

## What the curated sets look like on this assay

In [3]:
per_set <- do.call(rbind, lapply(split(p, p$set), function(d) data.frame(
  set = d$set[1], genes = length(unique(d$gene)),
  on_menu = length(unique(d$gene[d$on_menu])), probes = sum(d$on_menu))))
print(per_set, row.names = FALSE)
cat("\n-- IRG-28 --\n")
print(p[p$set == "IRG-28", c("gene", "SeqId", "Target", "column", "on_menu")], row.names = FALSE)
cat("\n-- NF-kB-11 control --\n")
print(p[p$set == "NF-kB-11", c("gene", "SeqId", "Target", "column", "on_menu")], row.names = FALSE)

      set genes on_menu probes
   IRG-28    28      11     12
 NF-kB-11    11       8     13



-- IRG-28 --


    gene        SeqId        Target            column on_menu
  CXCL10  seq.4141.79         IP-10            CXCL10    TRUE
   DDX60         <NA>          <NA>              <NA>   FALSE
  EPSTI1         <NA>          <NA>              <NA>   FALSE
    GBP1 seq.15326.64          GBP1              GBP1    TRUE
   HERC5  seq.12934.1         HERC5             HERC5    TRUE
   HERC6         <NA>          <NA>              <NA>   FALSE
   IFI27         <NA>          <NA>              <NA>   FALSE
   IFI44         <NA>          <NA>              <NA>   FALSE
  IFI44L         <NA>          <NA>              <NA>   FALSE
    IFI6         <NA>          <NA>              <NA>   FALSE
   IFIT1         <NA>          <NA>              <NA>   FALSE
   IFIT2   seq.9853.3         IFIT2             IFIT2    TRUE
   IFIT3 seq.13642.90         CIG49             IFIT3    TRUE
   IFIT5         <NA>          <NA>              <NA>   FALSE
   ISG15  seq.14148.2          UCRP ISG15_seq.14148.2    TRUE
   ISG15


-- NF-kB-11 control --


  gene        SeqId                            Target            column on_menu
 AICDA         <NA>                              <NA>              <NA>   FALSE
 CCND2         <NA>                              <NA>              <NA>   FALSE
  EBI3 seq.10851.77                             IL27B              EBI3    TRUE
  GZMB seq.14041.13                        Granzyme B GZMB_seq.14041.13    TRUE
  GZMB  seq.4133.54                        Granzyme B  GZMB_seq.4133.54    TRUE
  IFNG seq.15346.31                             IFN-g IFNG_seq.15346.31    TRUE
  IFNG  seq.2989.17                             IFN-g  IFNG_seq.2989.17    TRUE
  MSR1  seq.11207.3  Macrophage scavenger receptor:CD  MSR1_seq.11207.3    TRUE
  MSR1 seq.11282.16 Macrophage scavenger receptor:ECD MSR1_seq.11282.16    TRUE
  MSR1 seq.15533.97 Macrophage scavenger receptor:ECD MSR1_seq.15533.97    TRUE
  MSR1  seq.3684.78 Macrophage scavenger receptor:ECD  MSR1_seq.3684.78    TRUE
  SELL   seq.4831.4                     

**Explanation**
- `split(p, p$set)` divides the table into the 28 interferon response genes and the 11 controls;
  `lapply(..., function(d) ...)` makes one summary row for each, and `do.call(rbind, ...)` stacks
  them.
- `length(unique(d$gene[d$on_menu]))` counts genes, `sum(d$on_menu)` counts reagents. The two differ
  wherever a gene has more than one SOMAmer.
- The two printed tables are the audit: every curated gene, and either the reagent that measures it
  or `FALSE`.

**Result.** 11 of the 28 interferon response genes are on this menu, carrying 12 reagents: CXCL10,
GBP1, HERC5, IFIT2, IFIT3, ISG15 (two reagents, `seq.14148.2` and `seq.14151.4`), MX1, OAS1, OASL,
RTP4 and SIGLEC1. The 17 that are absent are absent because SomaScan v4.1 does not carry them:
DDX60, EPSTI1, HERC6, IFI27, IFI44, IFI44L, IFI6, IFIT1, IFIT5, LAMP3, LY6E, OAS2, OAS3, RSAD2,
SOCS1, SPATS2L, USP18. Several of those are the strongest genes in the transcriptomic score, so the
plasma proteome sees a smaller and different part of the interferon response than the blood
transcriptome does. That is a property of the assay, not a result.

The control survives the same filter: 8 of the 11 NF-kB-only genes are on the menu, carrying 13
reagents (EBI3, GZMB ×2, IFNG ×2, MSR1 ×4, SELL, SELP, TANK, TLR2). AICDA, CCND2 and XIAP are
absent. A control with 13 reagents is usable.

## Save

In [4]:
write.csv(p, art("ifn_panel.csv"), row.names = FALSE)
cat("wrote", art("ifn_panel.csv"), "\n")
cat(sprintf("%d rows: %d reagents on the menu, %d curated genes absent\n",
            nrow(p), sum(p$on_menu), sum(!p$on_menu)))

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-proteomics/data/run_artifacts/ifn_panel.csv 


45 rows: 25 reagents on the menu, 20 curated genes absent


**Explanation**
- `write.csv(..., row.names = FALSE)` writes the whole table, absent genes included, so a later
  notebook can state what the assay does not carry without re-reading `feature_metadata.txt`.
- `art("ifn_panel.csv")` puts it in `data/run_artifacts/`, which is gitignored: a run regenerates it
  from `genes/` and the study download, both of which are the actual inputs.

## Provenance

In [5]:
# Provenance. The rendered notebook is committed, so it records where it ran.
si <- Sys.info()
cat("run on      :", si[["nodename"]], "(", si[["sysname"]], si[["release"]], ")\n")
cat("date        :", format(Sys.time(), "%Y-%m-%d %H:%M %Z"), "\n")
cat("R           :", R.version.string, "|", R.version$platform, "\n")
cat("R comes from:", R.home(), "\n")
cat("packages    :\n")
for (p in c("WGCNA", "ComplexHeatmap", "circlize", "ReactomePA", "reactome.db",
            "org.Hs.eg.db", "sva", "VarSelLCM"))
  cat(sprintf("   %-16s %s\n", p,
              tryCatch(as.character(packageVersion(p)), error = function(e) "not installed")))

run on      : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )


date        : 2026-09-28 02:34 EDT 


R           : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 


R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics/lib/R 


packages    :


   WGCNA            1.74
   ComplexHeatmap   2.22.0
   circlize         0.4.18
   ReactomePA       1.50.0
   reactome.db      1.89.0
   org.Hs.eg.db     3.20.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
